# Sprout Humanoid - Environment Visualization

This notebook loads the `SproutMaintainVelocity` environment, renders the model
from multiple camera angles, runs a short rollout with zero/random actions, and
generates videos and snapshot sequences for visual confirmation.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import jax
import jax.numpy as jp
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import numpy as np
from tqdm import tqdm

from vnl_playground import tasks

# Enable persistent compilation cache to speed up repeated runs
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

print(f"JAX devices: {jax.devices()}")
print(f"Available environments: {list(tasks._envs.keys())}")

## 1. Load Environment and Inspect Model

In [ ]:
env = tasks.load("SproutMaintainVelocity", flatten_obs=False)

print(f"Action size (nu): {env.action_size}")
print(f"Num qpos (nq):    {env.mj_model.nq}")
print(f"Num qvel (nv):    {env.mj_model.nv}")
print(f"Num bodies:       {env.mj_model.nbody}")
print(f"Num joints:       {env.mj_model.njnt}")
print(f"Sim dt:           {env._config.sim_dt}")
print(f"Ctrl dt:          {env._config.ctrl_dt}")
print(f"Target speed:     {env._config.target_speed}")
print()
print("Joint names:")
for i, name in enumerate(env.get_joint_names()):
    print(f"  {i:2d}: {name}")

## 2. Static Renders - Standing Pose from Multiple Angles

Render the initial standing pose from multiple viewpoints to confirm the model
is loaded correctly and the standing pose looks reasonable.

In [ ]:
# Create MjData and set standing pose
mj_model = env.mj_model
mj_data = mujoco.MjData(mj_model)
mj_data.qpos[:] = np.array(env._init_qpos)
mujoco.mj_forward(mj_model, mj_data)

# Renderer
height, width = 480, 640
renderer = mujoco.Renderer(mj_model, height=height, width=width)

# Scene options
scene_option = mujoco.MjvOption()
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_JOINT] = True

# Define virtual camera positions: (azimuth, elevation, distance, lookat)
camera_configs = {
    "Front": dict(azimuth=180, elevation=-15, distance=2.5, lookat=[0, 0, 0.6]),
    "Right Side": dict(azimuth=90, elevation=-15, distance=2.5, lookat=[0, 0, 0.6]),
    "Back": dict(azimuth=0, elevation=-15, distance=2.5, lookat=[0, 0, 0.6]),
    "Left Side": dict(azimuth=270, elevation=-15, distance=2.5, lookat=[0, 0, 0.6]),
    "Top Down": dict(azimuth=180, elevation=-90, distance=2.5, lookat=[0, 0, 0.6]),
    "3/4 View": dict(azimuth=135, elevation=-25, distance=2.5, lookat=[0, 0, 0.6]),
}

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, (name, cam_cfg) in zip(axes.flat, camera_configs.items()):
    cam = mujoco.MjvCamera()
    cam.azimuth = cam_cfg["azimuth"]
    cam.elevation = cam_cfg["elevation"]
    cam.distance = cam_cfg["distance"]
    cam.lookat[:] = cam_cfg["lookat"]
    renderer.update_scene(mj_data, camera=cam, scene_option=scene_option)
    frame = renderer.render()
    ax.imshow(frame)
    ax.set_title(name, fontsize=14)
    ax.axis("off")

plt.suptitle("Sprout Humanoid - Standing Pose", fontsize=16)
plt.tight_layout()
plt.show()

renderer.close()

## 3. Reset and Step Test

Run `reset` and `step` through JAX to verify the MJX pipeline works end-to-end.

In [ ]:
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

print("JIT compiling reset... (this may take a few minutes on first run)")
state = jit_reset(jax.random.PRNGKey(0))
print(f"  Obs keys: {list(state.obs.keys())}")
print(f"  Reward:   {state.reward}")
print(f"  Done:     {state.done}")
print(f"  Torso z:  {state.data.qpos[2]:.4f}")

print("\nJIT compiling step...")
action = jp.zeros(env.action_size)
next_state = jit_step(state, action)
print(f"  Reward:   {next_state.reward}")
print(f"  Done:     {next_state.done}")
print(f"  Torso z:  {next_state.data.qpos[2]:.4f}")
print("\nReset and step working.")

## 4. Rollout with Zero Actions (Free Fall / Balance Test)

Apply zero torques and watch the robot fall under gravity. This confirms
the physics simulation is active and the termination criteria work.

In [ ]:
N_STEPS = 300

rng = jax.random.PRNGKey(42)
state = jit_reset(rng)

rollout_zero = [state]
rewards_zero = [float(state.reward)]
torso_z_zero = [float(state.data.qpos[2])]
dones_zero = [float(state.done)]

action = jp.zeros(env.action_size)
for i in tqdm(range(N_STEPS), desc="Zero-action rollout"):
    state = jit_step(state, action)
    rollout_zero.append(state)
    rewards_zero.append(float(state.reward))
    torso_z_zero.append(float(state.data.qpos[2]))
    dones_zero.append(float(state.done))

print(f"Final torso z: {torso_z_zero[-1]:.4f}")
print(f"Episode terminated: {dones_zero[-1] > 0}")

# Find first termination
term_step = next((i for i, d in enumerate(dones_zero) if d > 0), None)
if term_step is not None:
    print(f"Terminated at step {term_step} (torso z = {torso_z_zero[term_step]:.4f})")

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))

ax1.plot(rewards_zero)
ax1.set_title("Reward (zero actions)")
ax1.set_xlabel("Step")
ax1.set_ylabel("Reward")
ax1.grid(True, alpha=0.3)

ax2.plot(torso_z_zero)
ax2.axhline(0.3, color="red", linestyle="--", label="min_torso_z")
ax2.set_title("Torso Height (zero actions)")
ax2.set_xlabel("Step")
ax2.set_ylabel("Z (m)")
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3.plot(dones_zero)
ax3.set_title("Done Signal")
ax3.set_xlabel("Step")
ax3.set_ylabel("Done")
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Video - Zero Action Rollout

Render the zero-action rollout as a video to see the robot falling.

In [ ]:
def render_trajectory(env, trajectory, height=480, width=640, camera_cfg=None):
    """Render a trajectory of mjx_env.State objects to frames.

    Args:
        env: The environment instance.
        trajectory: List of mjx_env.State objects.
        height: Frame height.
        width: Frame width.
        camera_cfg: Optional dict with azimuth, elevation, distance, lookat.
            If None, uses the default free camera.

    Returns:
        List of numpy arrays (frames).
    """
    mj_model = env.mj_model
    mj_data = mujoco.MjData(mj_model)
    renderer = mujoco.Renderer(mj_model, height=height, width=width)

    if camera_cfg is not None:
        cam = mujoco.MjvCamera()
        cam.azimuth = camera_cfg.get("azimuth", 180)
        cam.elevation = camera_cfg.get("elevation", -15)
        cam.distance = camera_cfg.get("distance", 2.5)
        cam.lookat[:] = camera_cfg.get("lookat", [0, 0, 0.6])
    else:
        cam = -1  # Default free camera

    frames = []
    for state in tqdm(trajectory, desc="Rendering"):
        mj_data.qpos[:] = np.array(state.data.qpos)
        mj_data.qvel[:] = np.array(state.data.qvel)
        mujoco.mj_forward(mj_model, mj_data)
        renderer.update_scene(mj_data, camera=cam)
        frames.append(renderer.render().copy())

    renderer.close()
    return frames


# Render every 2nd frame to keep video manageable
render_every = 2
traj = rollout_zero[::render_every]
fps = 1.0 / env._config.ctrl_dt / render_every

cam_3_4 = dict(azimuth=135, elevation=-20, distance=2.5, lookat=[0, 0, 0.5])
frames_zero = render_trajectory(env, traj, camera_cfg=cam_3_4)

print(f"Rendered {len(frames_zero)} frames at {fps:.1f} fps")
media.show_video(frames_zero, fps=fps, loop=False)

## 6. Rollout with Random Actions

Apply small random torques to see the robot move in an uncoordinated way.
This confirms actuators are connected and responsive.

In [ ]:
N_STEPS = 500
ACTION_SCALE = 0.1  # Small random actions

rng = jax.random.PRNGKey(123)
state = jit_reset(rng)

rollout_random = [state]
rewards_random = [float(state.reward)]
torso_z_random = [float(state.data.qpos[2])]

for i in tqdm(range(N_STEPS), desc="Random-action rollout"):
    rng, act_rng = jax.random.split(rng)
    action = jax.random.normal(act_rng, shape=(env.action_size,)) * ACTION_SCALE
    state = jit_step(state, action)
    rollout_random.append(state)
    rewards_random.append(float(state.reward))
    torso_z_random.append(float(state.data.qpos[2]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(rewards_random)
ax1.set_title("Reward (random actions)")
ax1.set_xlabel("Step")
ax1.grid(True, alpha=0.3)

ax2.plot(torso_z_random)
ax2.axhline(0.3, color="red", linestyle="--", label="min_torso_z")
ax2.set_title("Torso Height (random actions)")
ax2.set_xlabel("Step")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
traj = rollout_random[::render_every]
frames_random = render_trajectory(env, traj, camera_cfg=cam_3_4)

print(f"Rendered {len(frames_random)} frames")
media.show_video(frames_random, fps=fps, loop=False)

## 7. Snapshot Sequences

Show key frames from the rollouts as image strips for quick visual comparison.

In [ ]:
def show_snapshot_strip(frames, title, n_snapshots=8):
    """Display evenly-spaced frames as a horizontal strip."""
    indices = np.linspace(0, len(frames) - 1, n_snapshots, dtype=int)
    fig, axes = plt.subplots(1, n_snapshots, figsize=(3 * n_snapshots, 4))
    for ax, idx in zip(axes, indices):
        ax.imshow(frames[idx])
        ax.set_title(f"Frame {idx}", fontsize=10)
        ax.axis("off")
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


show_snapshot_strip(frames_zero, "Zero Actions - Falling Sequence")
show_snapshot_strip(frames_random, "Random Actions - Uncoordinated Movement")

## 8. Multi-Angle Video

Render the random-action rollout from 4 angles side-by-side.

In [ ]:
traj_short = rollout_random[:200:render_every]

angle_configs = {
    "Front": dict(azimuth=180, elevation=-15, distance=2.5, lookat=[0, 0, 0.5]),
    "Side": dict(azimuth=90, elevation=-15, distance=2.5, lookat=[0, 0, 0.5]),
    "3/4": dict(azimuth=135, elevation=-25, distance=2.5, lookat=[0, 0, 0.5]),
    "Top": dict(azimuth=180, elevation=-80, distance=2.5, lookat=[0, 0, 0.5]),
}

h, w = 240, 320
all_angle_frames = {}
for name, cfg in angle_configs.items():
    print(f"Rendering {name}...")
    all_angle_frames[name] = render_trajectory(
        env, traj_short, height=h, width=w, camera_cfg=cfg
    )

# Stitch into 2x2 grid
n_frames = len(traj_short)
grid_frames = []
names = list(angle_configs.keys())
for i in range(n_frames):
    top = np.concatenate(
        [all_angle_frames[names[0]][i], all_angle_frames[names[1]][i]], axis=1
    )
    bottom = np.concatenate(
        [all_angle_frames[names[2]][i], all_angle_frames[names[3]][i]], axis=1
    )
    grid_frames.append(np.concatenate([top, bottom], axis=0))

print(f"Grid video: {grid_frames[0].shape}")
media.show_video(grid_frames, fps=fps, loop=False)

## 9. Save Videos to Disk

In [ ]:
output_dir = "sprout_videos"
os.makedirs(output_dir, exist_ok=True)

media.write_video(f"{output_dir}/sprout_zero_actions.mp4", frames_zero, fps=fps, qp=18)
media.write_video(
    f"{output_dir}/sprout_random_actions.mp4", frames_random, fps=fps, qp=18
)
media.write_video(
    f"{output_dir}/sprout_multi_angle.mp4", grid_frames, fps=fps, qp=18
)

print(f"Videos saved to {output_dir}/")
for f in os.listdir(output_dir):
    size_mb = os.path.getsize(f"{output_dir}/{f}") / 1e6
    print(f"  {f}: {size_mb:.1f} MB")

## 10. Reward and Termination Diagnostics

Inspect individual reward components and termination metrics from the rollout.

In [ ]:
# Run a diagnostic rollout that collects metrics
rng = jax.random.PRNGKey(99)
state = jit_reset(rng)

metric_keys = None
metric_history = {}

for i in tqdm(range(300), desc="Diagnostic rollout"):
    rng, act_rng = jax.random.split(rng)
    action = jax.random.normal(act_rng, shape=(env.action_size,)) * 0.05
    state = jit_step(state, action)

    if metric_keys is None:
        metric_keys = [k for k in state.metrics.keys()]
        for k in metric_keys:
            metric_history[k] = []

    for k in metric_keys:
        metric_history[k].append(float(state.metrics[k]))

# Plot reward components
reward_keys = [k for k in metric_keys if k.startswith("rewards/")]
term_keys = [k for k in metric_keys if k.startswith("terminations/")]

if reward_keys:
    fig, axes = plt.subplots(1, len(reward_keys), figsize=(5 * len(reward_keys), 4))
    if len(reward_keys) == 1:
        axes = [axes]
    for ax, k in zip(axes, reward_keys):
        ax.plot(metric_history[k])
        ax.set_title(k)
        ax.set_xlabel("Step")
        ax.grid(True, alpha=0.3)
    plt.suptitle("Reward Components", fontsize=14)
    plt.tight_layout()
    plt.show()

if term_keys:
    fig, axes = plt.subplots(1, len(term_keys), figsize=(5 * len(term_keys), 4))
    if len(term_keys) == 1:
        axes = [axes]
    for ax, k in zip(axes, term_keys):
        ax.plot(metric_history[k])
        ax.set_title(k)
        ax.set_xlabel("Step")
        ax.grid(True, alpha=0.3)
    plt.suptitle("Termination Signals", fontsize=14)
    plt.tight_layout()
    plt.show()

print("\nAll metrics at final step:")
for k in metric_keys:
    print(f"  {k}: {metric_history[k][-1]:.6f}")